# 03 · Evaluate the video transformer on the test set

Runs the test recordings through every final model of `01_train_video_transformer`. Nothing is fitted or chosen here:

1. **Test similarities** of all six subjects for every setting (test EEG normalised with the statistics of the training recordings; CLIP targets of the 102 test videos, centred across the candidates).
2. **Test scores** (top-1, top-5, mean rank) for sub-01 to sub-04, whose test labels exist, next to their validation scores from `01`.
3. **Submission** for sub-05 and sub-06 from the setting that `01` chose on validation (`models/chosen.json`), plus sub-01 with the same code to check the format, as the task asks. The test scores of the other settings are reported for completeness; they do not change the choice.
4. **Failure cases** of the chosen model: where the true video ends up, and whether the same videos are hard for every subject.

**Saved** next to the evaluated models: `video_transformer/<run of 01>/test/` (tables, figures, similarity matrices, `submission/`).

In [ ]:
from pathlib import Path
import json, os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import decimate
from tqdm import tqdm

## 1. Configuration

In [ ]:
DATA_ROOT = Path(os.environ.get('EEG_DATA_ROOT', str(Path.home() / 'eeg-data' / 'eeg-mind-reading-challenge'))).expanduser()
if (DATA_ROOT / 'eeg-mind-reading-challenge').is_dir():
    DATA_ROOT /= 'eeg-mind-reading-challenge'
VIDEO_CLIP_DIR = Path(os.environ.get('EEG_EMBEDDING_ROOT', str(Path.home() / 'eeg-data' / 'eeg-video-embeddings'))
                      ).expanduser() / 'clip_vit_b32'          # CLIP ViT-B/32 per frame of our videos (from 04)
RESULTS_ROOT = Path(os.environ.get('EEG_RESULTS_ROOT', str(Path.home() / 'eeg-results'))).expanduser()
OUT_ROOT = RESULTS_ROOT / 'pretrained-transformers' / 'video_transformer'
TRAIN_RUN = None                     # run of 01 to evaluate (e.g. '2026-10-19_101500'); None = the latest finished one

SUBJECTS = [1, 2, 3, 4, 5, 6]
DEV_SUBJECTS = [1, 2, 3, 4]          # test labels exist (tst_stim): scored here
FINAL_SUBJECTS = [5, 6]              # labels withheld: the submission
TEST_VIDEO_IDS = np.arange(1001, 1103)
N_CANDIDATES = len(TEST_VIDEO_IDS)
DEVICE = 'cuda:0'                    # inference only: one GPU is enough

finished = sorted(p.parent.parent for p in OUT_ROOT.glob('*/models/chosen.json'))
assert TRAIN_RUN or finished, f'no finished run of 01 in {OUT_ROOT}'
TRAIN_DIR = OUT_ROOT / TRAIN_RUN if TRAIN_RUN else finished[-1]
TEST_DIR = TRAIN_DIR / 'test'
for d in [TEST_DIR / 'similarity', TEST_DIR / 'submission', TEST_DIR / 'figures']:
    d.mkdir(parents=True, exist_ok=True)
CONFIG = json.loads((TRAIN_DIR / 'config.json').read_text())
CHOSEN = json.loads((TRAIN_DIR / 'models' / 'chosen.json').read_text())
MODELS = [json.loads(p.read_text()) for p in sorted((TRAIN_DIR / 'models').glob('*.json')) if p.name != 'chosen.json']
print(f'run {TRAIN_DIR.name}: {len(MODELS)} models', [m['name'] for m in MODELS], '\nchosen on validation:', CHOSEN['name'])

## 2. Model and preprocessing (as in `01`)

The same architecture and EEG preprocessing as in `01`, with the sizes read from its `config.json`. The test EEG keeps its original row order and is normalised with each subject's **training** statistics, which are stored with the models.

In [ ]:
ARCH = CONFIG['architecture']
N_CHANNELS, N_SAMPLES, PATCH, D_MODEL, DEPTH, HEADS, DROPOUT, CLIP_DIM, N_SLOTS, MODEL_SFREQ, EEG_SFREQ = (ARCH[k] for k in [
    'n_channels', 'n_samples', 'patch', 'd_model', 'depth', 'heads', 'dropout', 'clip_dim', 'n_slots', 'model_sfreq',
    'eeg_sfreq'])
CHALLENGE_SLOT = {int(k): v for k, v in ARCH['challenge_slot'].items()}


class EEGTransformer(nn.Module):
    """As in 01; the initial values do not matter here, all weights are loaded."""

    def __init__(self):
        super().__init__()
        self.subject_mix = nn.Parameter(torch.zeros(N_SLOTS, N_CHANNELS, N_CHANNELS))
        self.subject_token = nn.Parameter(torch.zeros(N_SLOTS, D_MODEL))
        self.patch_embed = nn.Linear(PATCH, D_MODEL)
        self.channel_pos = nn.Parameter(torch.zeros(N_CHANNELS, 1, D_MODEL))
        self.time_pos = nn.Parameter(torch.zeros(1, N_SAMPLES // PATCH, D_MODEL))
        layer = nn.TransformerEncoderLayer(D_MODEL, HEADS, 4 * D_MODEL, DROPOUT, activation='gelu',
                                           batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, DEPTH, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, CLIP_DIM)
        self.logit_scale = nn.Parameter(torch.zeros(()))

    def forward(self, x, slot):                                                     # x: (batch, 17, 80)
        x = torch.bmm(self.subject_mix[slot], x)
        b, c, t = x.shape
        tokens = self.patch_embed(x.reshape(b, c, t // PATCH, PATCH))               # (batch, 17, 16, d)
        tokens = (tokens + self.channel_pos + self.time_pos).flatten(1, 2)          # (batch, 272, d)
        h = self.norm(self.encoder(torch.cat([self.subject_token[slot][:, None], tokens], dim=1))[:, 0])
        return F.normalize(self.head(h).float(), dim=-1)


def load_test_eeg(sub):
    """Test recordings (102, 17, 300) of one subject at 100 Hz, in the original order of tst/sub-XX_tst.npy."""
    eeg = np.load(DATA_ROOT / 'tst' / f'sub-{sub:02d}_tst.npy').astype(np.float64)
    eeg -= eeg.mean(axis=2, keepdims=True)
    return decimate(eeg, EEG_SFREQ // MODEL_SFREQ, axis=-1, ftype='fir', zero_phase=True)


def normalise(eeg, stats):
    """Whitening and z-score with the training statistics (W, mean, std) -> model inputs (n, 17, 80)."""
    W, mu, sd = (t.to(eeg.device) for t in stats)
    return ((torch.einsum('cd,ndt->nct', W, eeg)[:, :, :N_SAMPLES] - mu) / sd).float()


@torch.no_grad()
def embed(model, x, slot):
    with torch.autocast('cuda', dtype=torch.bfloat16):
        return model.eval()(x, slot)


def true_ranks(S, true_cols):
    """Rank of the true video (1 = first) in the ranked list a submission contains (stable sort)."""
    order = np.argsort(-np.asarray(S), axis=1, kind='stable')
    return np.argmax(order == np.asarray(true_cols)[:, None], axis=1) + 1


def metrics(S, true_cols):
    rank = true_ranks(S, true_cols)
    return dict(n=len(rank), top1=float(np.mean(rank == 1)), top5=float(np.mean(rank <= 5)), mean_rank=float(rank.mean()))


TEST_EEG = {sub: load_test_eeg(sub) for sub in tqdm(SUBJECTS, desc='Loading test EEG', unit='subject')}
LABELS = {sub: np.load(DATA_ROOT / 'tst_stim' / f'sub-{sub:02d}_tst_stim.npy').reshape(-1) for sub in DEV_SUBJECTS}
assert all(np.array_equal(np.sort(v), TEST_VIDEO_IDS) for v in LABELS.values())
TRUE_COLS = {sub: np.searchsorted(TEST_VIDEO_IDS, v) for sub, v in LABELS.items()}

with np.load(VIDEO_CLIP_DIR / 'embeddings.npz') as z:
    FRAME_EMB = {v: z[str(v)] for v in TEST_VIDEO_IDS}
FRAME_T = np.array(json.loads((VIDEO_CLIP_DIR / 'metadata.json').read_text())['frame_times_s'])
FRAME_CENTRE = FRAME_T + np.median(np.diff(FRAME_T)) / 2        # as in 01


def test_targets(window_s):
    """CLIP target of each test video (mean over the frames of the window, unit length), centred across the 102."""
    keep = (FRAME_CENTRE >= window_s[0]) & (FRAME_CENTRE < window_s[1])
    t = F.normalize(torch.as_tensor(np.stack([FRAME_EMB[v][keep].mean(0) for v in TEST_VIDEO_IDS]), device=DEVICE), dim=-1)
    return F.normalize(t - t.mean(dim=0, keepdim=True), dim=-1)

## 3. Test similarities of every model

Per setting: each of the 4 seeds embeds the test recordings; cosine similarity with the centred CLIP targets of the 102 test videos; mean over the seeds (the seed ensemble of `01`). Saved per setting and subject: rows = test recordings in their original order, columns = videos 1001–1102.

In [ ]:
SIMS = {}
for cfg in tqdm(MODELS, desc='Test similarities', unit='model'):
    saved = torch.load(TRAIN_DIR / 'models' / f"{cfg['name']}.pt", map_location=DEVICE, weights_only=True)
    models = []
    for seed in cfg['seeds']:
        model = EEGTransformer().to(DEVICE)
        model.load_state_dict(saved['states'][seed])
        models.append(model)
    cand = test_targets(cfg['video_window_s'])
    order = np.array(cfg['channel_index'])
    (TEST_DIR / 'similarity' / cfg['name']).mkdir(exist_ok=True)
    SIMS[cfg['name']] = {}
    for sub in SUBJECTS:
        x = normalise(torch.as_tensor(TEST_EEG[sub][:, order], device=DEVICE), saved['normalisation'][sub])
        slot = torch.full((len(x),), CHALLENGE_SLOT[sub], device=DEVICE)
        S = np.mean([(embed(m, x, slot) @ cand.T).cpu().numpy() for m in models], axis=0)
        SIMS[cfg['name']][sub] = S
        np.save(TEST_DIR / 'similarity' / cfg['name'] / f'sub-{sub:02d}_tst_similarity.npy', S)

## 4. Test scores (sub-01 to sub-04) next to validation

Chance: top-1 0.010, top-5 0.049. Validation = 5 folds of 102 held-out **training** videos (`01`), test = the 102 test videos. If the ranking of the settings agrees, the validation protocol was a sound basis for the choice.

In [ ]:
TEST = pd.DataFrame([dict(model=cfg['name'], subject=sub, **metrics(SIMS[cfg['name']][sub], TRUE_COLS[sub]))
                     for cfg in MODELS for sub in DEV_SUBJECTS])
VAL = pd.DataFrame([dict(model=c['name'], val_top1=c['validation']['top1'], val_top5=c['validation']['top5'])
                    for c in MODELS]).set_index('model')
SUMMARY = (TEST.groupby('model')[['top1', 'top5', 'mean_rank']].mean().join(VAL)
           .sort_values('val_top5', ascending=False))
SUMMARY['chosen'] = SUMMARY.index == CHOSEN['name']
TEST.to_csv(TEST_DIR / 'test_per_subject.csv', index=False)
SUMMARY.reset_index().to_csv(TEST_DIR / 'test_summary.csv', index=False)

x = np.arange(len(SUMMARY))
per_sub = TEST.pivot(index='model', columns='subject', values='top5').loc[SUMMARY.index]
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.bar(x - 0.2, SUMMARY.val_top5, 0.4, color='0.7', label='validation (held-out training videos)')
ax.bar(x + 0.2, SUMMARY.top5, 0.4, color='C0', label='test (sub-01 to sub-04)')
for i, vals in enumerate(per_sub.values):
    ax.plot(np.full(len(vals), i + 0.2), vals, 'k.', ms=3)
ax.axhline(5 / N_CANDIDATES, color='0.4', ls='--', lw=1, label='chance')
ax.set_xticks(x)
ax.set_xticklabels([n + ('  [chosen]' if c else '') for n, c in zip(SUMMARY.index, SUMMARY.chosen)],
                   rotation=40, ha='right', fontsize=7)
ax.set_ylabel('top-5')
ax.set_title('Video transformer: validation vs test (dots = test subjects)', fontsize=10)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(TEST_DIR / 'figures' / 'validation_vs_test.png', dpi=130, bbox_inches='tight')
plt.show()
SUMMARY.round(3)

## 5. Submission (setting chosen on validation in `01`)

`sub-XX_ranked_candidate_list.npy`: 102 × 102 integers. Row *i* belongs to the *i*-th recording of `tst/sub-XX_tst.npy` in its original order and holds all video IDs 1001–1102, from most to least likely. sub-01 is written with the same code and scored from the file itself, to check the format.

In [ ]:
for sub in FINAL_SUBJECTS + [DEV_SUBJECTS[0]]:
    ranked = TEST_VIDEO_IDS[np.argsort(-SIMS[CHOSEN['name']][sub], axis=1, kind='stable')].astype(int)
    assert ranked.shape == (102, 102) and all(np.array_equal(np.sort(r), TEST_VIDEO_IDS) for r in ranked)
    np.save(TEST_DIR / 'submission' / f'sub-{sub:02d}_ranked_candidate_list.npy', ranked)

check = np.load(TEST_DIR / 'submission' / 'sub-01_ranked_candidate_list.npy')
place = np.array([np.flatnonzero(row == v)[0] for row, v in zip(check, LABELS[1])])       # 0 = first place
m = metrics(SIMS[CHOSEN['name']][1], TRUE_COLS[1])
print(f"format check, sub-01 scored from its file: top-1 {np.mean(place == 0):.3f}, top-5 {np.mean(place < 5):.3f} "
      f"(from the similarities: {m['top1']:.3f}, {m['top5']:.3f}; must be equal)")
print('written:', sorted(p.name for p in (TEST_DIR / 'submission').glob('*.npy')))

## 6. Failure cases of the chosen model

- **Top-k curve.** The share of recordings whose true video is among the first *k* places, for every *k* (top-1 and top-5 are two points of it). Chance is the diagonal *k*/102.
- **Same videos hard for every subject?** Each subject saw each test video once. If the failures were only single-trial noise, the rank of a video in one subject would say nothing about its rank in another (Spearman *r* ≈ 0; with 102 videos, |*r*| > 0.2 is unlikely by chance, p < 0.05). A positive *r* means some videos are systematically hard, a property of the stimulus. Their IDs are listed and saved, to look at the videos (frames are not put into the repository, by the challenge rules).

In [ ]:
S_CHOSEN = SIMS[CHOSEN['name']]
RANK = pd.DataFrame({f'sub-{sub:02d}': pd.Series(true_ranks(S_CHOSEN[sub], TRUE_COLS[sub]), index=LABELS[sub])
                     for sub in DEV_SUBJECTS}).sort_index()                   # rows: test videos, values: rank
RANK['mean'] = RANK.mean(axis=1)
RANK.rename_axis('video').reset_index().to_csv(TEST_DIR / 'failure_cases_rank_per_video.csv', index=False)

k = np.arange(1, N_CANDIDATES + 1)
fig, ax = plt.subplots(figsize=(6, 3.8))
for col in RANK.columns[:-1]:
    ax.plot(k, [(RANK[col] <= kk).mean() for kk in k], lw=1.2, label=col)
ax.plot(k, k / N_CANDIDATES, 'k--', lw=1, label='chance')
ax.set_xscale('log')
ax.set_xlabel('k (log scale)')
ax.set_ylabel('share of recordings with the true video in the top k')
ax.set_title(f"Top-k curve: {CHOSEN['name']}", fontsize=10)
ax.legend(fontsize=8)
fig.savefig(TEST_DIR / 'figures' / 'failure_topk_curve.png', dpi=130, bbox_inches='tight')
plt.show()

r = RANK.drop(columns='mean').corr(method='spearman').values[np.triu_indices(len(DEV_SUBJECTS), 1)]
print(f'rank of the same video across subjects: mean pairwise Spearman r = {r.mean():.2f} '
      f'(pairs from {r.min():.2f} to {r.max():.2f})')
print('easiest videos (mean rank over sub-01..04):', RANK['mean'].nsmallest(5).round(1).to_dict())
print('hardest videos:', RANK['mean'].nlargest(5).round(1).to_dict())

**Reference result** (same pipeline in `pretrained_transformers/05`, run 2026-10-18; chosen: `finetune`, causal 0–0.7 s window): test top-1 0.103, top-5 0.324, mean rank 16.8 (sub-01: 0.088 / 0.314, sub-02: 0.127 / 0.382, sub-03: 0.088 / 0.304, sub-04: 0.108 / 0.294); chance 0.010 / 0.049. The test scores were higher than validation (top-5 0.229); the final models are trained on ~11 % more videos (900 instead of ~810 after the inner split), and the test videos are a different sample of 102.